In [ ]:
#  Create Spark Session with custom configurations
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("Bucket concept & joins")
    .master("local[*]")
    .config("spark.executor.cores",4)
    .config("spark.core.max",16)
    .config("spark.executor.memory","512M")
    .getOrCreate()
)

In [16]:
spark

In [65]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [66]:
# reading the big table -employee_rec
_schema="first_name string,last_name string,job_title string,dob string,email string,phone string,salary string,department_id string"
emp=spark.read.format("csv").schema(_schema).option("header",True).load("employee_rec.csv")

In [ ]:
# reading small table - Department_data
_schema="department_id string,department_name string,description string,city string,state string,country string"
dept=spark.read.format("csv").schema(_schema).option("header",True).load("department_data.csv")

In [ ]:
# Perform normal LEFT OUTER JOIN (broadcast disabled)
df_joined=emp.join(dept, on=emp.department_id==dept.department_id,how="left_outer")

In [ ]:
# Save Dataframe to csv
df_joined.write.format("noop").mode("overwrite").save()

In [ ]:
# Broadcast join
from pyspark.sql.functions import broadcast
df_joined=emp.join(broadcast(dept),on=emp.department_id==dept.department_id,how="left_outer")

In [71]:
df_joined.write.format("noop").mode("overwrite").save()

In [72]:
# join big & big table

In [73]:
# read the sales data
sales_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"
sales=spark.read.format("csv").schema(sales_schema).option("header",True).load("new_sales.csv")

In [74]:
# read the cities data
city_schema="city_id string, city string, state string, state_abv string, country string"
city=spark.read.format("csv").schema(city_schema).option("header",True).load("cities.csv")

In [ ]:
# Performing left outer join on two big tables
sales_joined=sales.join(city,on=sales.city_id==city.city_id,how="left_outer")
sales_joined.write.format("noop").mode("overwrite").save()

In [76]:
sales_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [city_id#772], [city_id#779], LeftOuter
:- *(1) Sort [city_id#772 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(city_id#772, 200), ENSURE_REQUIREMENTS, [id=#608]
:     +- FileScan csv [transacted_at#767,trx_id#768,retailer_id#769,description#770,amount#771,city_id#772] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jupyter/new_sales.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit...
+- *(3) Sort [city_id#779 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(city_id#779, 200), ENSURE_REQUIREMENTS, [id=#620]
      +- *(2) Filter isnotnull(city_id#779)
         +- FileScan csv [city_id#779,city#780,state#781,state_abv#782,country#783] Batched: false, DataFilters: [isnotnull(city_id#779)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jupyter/cities.csv], Partit

In [80]:
# write sales and city data in buckets
sales.write.format("csv").mode("overwrite").bucketBy(4,"city_id").option("header",True).option("path","/home/jupyter/new_sales_data.csv").saveAsTable("sales_bucket")
# (
#     sales.write
#         .mode("overwrite")
#         .bucketBy(4, "city_id")
#         .sortBy("city_id")
#         .saveAsTable("sales_bucket")
# )

In [83]:
# city.write.format("csv").mode("overwrite").bucketBy(4,"city_id").option("header",True).option("path","/home/jupyter/cities.csv").saveAsTable("cities_bucket")
(
     city.write
         .mode("overwrite")
         .bucketBy(4, "city_id")
         .sortBy("city_id")
         .saveAsTable("city_bucket")
)

In [85]:
spark.sql("show tables in default").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default| city_bucket|      false|
|  default|sales_bucket|      false|
+---------+------------+-----------+



In [86]:
sales_bucket=spark.read.table("sales_bucket")

In [88]:
city_bucket=spark.read.table("city_bucket")

In [ ]:
# Join bucketed tables (Bucket Join optimization)
df_joined=sales_bucket.join(city_bucket,on=sales_bucket.city_id==city_bucket.city_id,how="left_outer")

In [ ]:
df_joined.write.format("noop").mode("overwrite").save()